# Rae2822 2D Wing Slice Model

### Import Library

In [ ]:
import pandas as pd
import glob
import matplotlib.pyplot as plt
import numpy as np

### Loading Data

In [ ]:
# angles = range(21)

folder_path = "./data/rae2822/angvar_sa/"
angles = sorted(glob.glob(f"{folder_path}*"), key=lambda x: float(x.split('\\')[-1]))
files = sorted(glob.glob(f"{folder_path}*/surface_flow.csv"))

master_df = []

for i, filepath in enumerate(angles):
    df = pd.read_csv(filepath+"/surface_flow.csv")

    # apply angles for each case
    df['AoA'] = float(angles[i].split('\\')[-1])

    master_df.append(df)

# Combine into one training set
full_data = pd.concat(master_df, ignore_index=True)
full_data = full_data[['Pressure_Coefficient', 'AoA', 'x', 'y']]
# full_data = full_data.dropna()
print(full_data[full_data.isna().any(axis=1)])


In [ ]:
print(f"Total rows: {len(full_data)}")
full_data[['Pressure_Coefficient', 'AoA', 'x', 'y']].head()

In [ ]:
sample_case = full_data[full_data['AoA'] == 0.0]

plt.figure(figsize=(10, 6))
top_surface = sample_case[sample_case['y'] > 0].sort_values(by='x')
bottom_surface = sample_case[sample_case['y'] <= 0].sort_values(by='x')

plt.scatter(top_surface['x'], top_surface['Pressure_Coefficient'], color='purple', label='(Top Surface)')
plt.scatter(bottom_surface['x'], bottom_surface['Pressure_Coefficient'],color='orange', label='(Bottom Surface)')
plt.gca().invert_yaxis()
plt.legend()
plt.grid(True)
plt.title("Sample Case: AoA = 0.0°")

# Model

In [ ]:
# import model
import tensorflow as tf
from sklearn.model_selection import LeaveOneGroupOut

print("TensorFlow version:", tf.__version__)

In [ ]:
features = ['x', 'y', 'AoA']
target = 'Pressure_Coefficient'

In [ ]:
logo = LeaveOneGroupOut()

architecture = [
        tf.keras.layers.Dense(64, activation='relu', input_shape=(3,)),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.Dense(1)
    ]

aoa_values = full_data['AoA'].values
X = full_data[features].values  # shape (n*192, 3)
y = full_data[target].values    # shape (n*192, 1)

val_scores = []
_round = 1
for train_idx, test_idx in logo.split(X, y, aoa_values):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    model = tf.keras.Sequential(architecture)

    model.compile(
        optimizer='adam',
        loss='mse',
        metrics=['accuracy']
    )

    model.fit(
        X_train, y_train,
        epochs=10,
        verbose=0
    )

    score = model.evaluate(X_test, y_test, verbose=0)
    val_scores.append(score)
    print(f"Round {_round} --> Fold MSE: {score}")
    _round += 1

print(f"\nMean MSE: {np.mean(val_scores):.4f}")

In [64]:
final_model = tf.keras.Sequential(architecture)
final_model.compile(
        optimizer='adam',
        loss='mse',
        metrics=['accuracy']
    )
final_model.fit(
        X, y,
        epochs=10,
        verbose=0
    )

predict_aoa = 25

X_new = full_data[full_data['AoA'] == 0.0][features].copy()
X_new = X_new.reset_index(drop=True)
# predict
Cp_predicted = final_model.predict(X_new)

# map back — just attach prediction to the same rows
result = X_new[['x', 'y', 'AoA']].copy()
result['Cp_predicted'] = Cp_predicted

print(result)

6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 
            x         y  AoA  Cp_predicted
0    0.985397  0.000943  0.0      0.279886
1    0.969573  0.001489  0.0      0.304464
2    0.952680  0.001679  0.0      0.319638
3    0.934996  0.001511  0.0      0.322523
4    0.916822  0.000923  0.0      0.320115
..        ...       ...  ...           ...
187  0.964163  0.007525  0.0      0.128670
188  0.973876  0.005604  0.0      0.160513
189  0.983087  0.003730  0.0      0.180000
190  0.991850  0.001925  0.0      0.242343
191  1.000000  0.000170  0.0      0.261445

[192 rows x 4 columns]
